[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hawksight-AI/semantica/blob/main/cookbook/use_cases/healthcare/01_Clinical_Intelligence_with_Semantica.ipynb)

# Clinical Intelligence with Semantica

**Building Explainable Healthcare Knowledge Systems**

Clinical decision support has a specific technical requirement that most RAG pipelines don't meet: every recommendation needs an inspectable chain of premises, not just a plausible-sounding answer. This notebook builds that requirement into a working system with [Semantica](https://github.com/Hawksight-AI/semantica), using three synthetic patient cases as running examples: (1) type 2 diabetes with hypertension and elevated HbA1c, (2) type 2 diabetes with declining renal function on a drug that becomes contraindicated below a specific eGFR threshold, and (3) asthma. A fourth case is introduced in the closing capstone to test that the pipeline generalizes.

We go through ingestion, ontology engineering, knowledge graph construction, reasoning, decision tracking, and export end to end, using Semantica's own classes at each step rather than hand-rolled substitutes.

### Two kinds of data, and why the difference matters

- **Real, live reference data.** Disease identity, definitions, and synonyms come from [Wikidata](https://www.wikidata.org/) and [EBI OLS4](https://www.ebi.ac.uk/ols4)/[MONDO](https://mondo.monarchinitiative.org/); drug identifiers from [RxNav/RxNorm](https://lhncbc.nlm.nih.gov/RxNav/); ICD-10-CM codes from [NLM Clinical Tables](https://clinicaltables.nlm.nih.gov/); literature from [PubMed via NCBI E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/); and drug-label contraindication language from [openFDA](https://open.fda.gov/), all fetched live, at runtime, through Semantica's `ingest` module (`OntologyIngestor`, `PublicAPIIngestor`, `WebIngestor`). There is no placeholder text anywhere in this pipeline: if a source that grounds a downstream rule or ontology is unreachable, the notebook raises a clear error rather than substituting fabricated content; only per-item enrichment (e.g. one disease's synonym list) degrades gracefully by being left empty.
- **Synthetic patients.** Three fictional cases (fictional MRNs, fictional names) carry the reasoning and decision narrative. **No real patient data appears anywhere in this notebook.** Live FHIR/MCP integration with a real medical database is a separate notebook, `05_Medical_Database_Integration.ipynb`.

This is notebook **01** in the healthcare use-case series, alongside `02_Disease_Network_Analysis.ipynb` (a deeper look at disease comorbidity networks) and `05_Medical_Database_Integration.ipynb`. Although every example below is a healthcare one, none of the underlying mechanics are healthcare-specific. The closing "Adapting This Pattern to Other Domains" section maps each Part onto its domain-independent building block.

## Prerequisites

- **Live network access is required, not optional.** Five of the six data sources in Part 2 (Wikidata, EBI OLS4/MONDO, openFDA, RxNav, NLM Clinical Tables) raise a `RuntimeError` and stop the notebook if unreachable; only PubMed degrades gracefully.
- **Expected runtime:** roughly 2-5 minutes end to end on a typical connection; longer if any of the six public APIs rate-limit or respond slowly.
- **First-time install size:** `pip install -qU semantica` pulls in `torch`, `transformers`, `spacy`, and `fastembed` among other dependencies — a multi-GB download the first time it's run in a fresh environment.

## Installation

```bash
pip install semantica
```

SHACL validation support (`pyshacl`) is tracked in [#736](https://github.com/semantica-agi/semantica/issues/736) — install `pyshacl` directly for now if you want to try live validation in Part 6:
```bash
pip install pyshacl
```

Every external call in this notebook is to a free, public, no-API-key API. No LLM key is needed to run it top to bottom.

In [ ]:
!pip install -qU semantica

## Part 1: Introduction

### Why healthcare needs explainable AI

When a clinical AI system recommends intensifying therapy or stopping a medication, the operative question isn't "how confident are you?" in the abstract. It's "show your work": which guideline, which lab value, which prior case this resembles, and what would have to change for the answer to flip. A recommendation without that trail isn't actionable in a regulated clinical setting.

### Where vector-similarity RAG falls short here

Embedding documents and retrieving the top-k most similar chunks is a genuinely useful technique, and this notebook uses it too (Part 8, Part 12). But used *alone* for clinical decision support, it runs into structural limits:

- **Similarity isn't clinical relevance.** A chunk about monitoring hypertension can be textually closer to a query than a chunk about a treatment contraindication, even when the contraindication is the one fact that matters.
- **"Highest cosine similarity" is not an audit trail.** It doesn't tell a reviewer *why* this particular passage justified the decision.
- **Nothing stops a bad recommendation from slipping through.** No constraint anywhere says "not this drug, this patient, this lab value."
- **There's no native concept of policy.** A vector index doesn't know that treatment-modification decisions require 0.90 confidence and a second reviewer.

### Why a knowledge graph and ontology layer closes the gap

Typed relationships make "related to" precise: `comorbid_with` is not `treated_with`. Rule-based reasoning derives new facts *and* keeps the premises attached. SHACL (Shapes Constraint Language, the W3C standard for validating RDF/graph data against a schema) constraints catch a missing or invalid relationship before it reaches a downstream consumer. A decision-tracking layer remembers precedent, causality, and policy compliance for every recommendation made. Vector search stays in the toolkit. It's excellent at finding candidates, but here it works *alongside* structure rather than in place of it.

### The reusable pattern behind this notebook

Every technique below is demonstrated on healthcare data, but none of it is healthcare-specific at the architecture level. The pipeline is:

`ingest real reference data` → `build/reuse an ontology` → `align to external identifiers` → `validate with constraints` → `build a knowledge graph` → `search, reason, and track decisions over it` → `export to standard formats`

Swap "disease/drug ontology aligned to ICD-10/MeSH/SNOMED" for "financial-instrument ontology aligned to a regulatory taxonomy," or "clinical guideline contraindication rule" for "loan-underwriting policy rule," and the same 14-part structure applies unchanged. The closing section of this notebook makes that mapping explicit for a few other domains.

### What we're building it with

[Semantica](https://github.com/Hawksight-AI/semantica) is a semantic intelligence and knowledge engineering framework. Rather than importing everything up front, each class from `semantica.ingest`, `semantica.parse`, `semantica.split`, `semantica.provenance`, `semantica.ontology`, `semantica.triplet_store`, `semantica.kg`, `semantica.semantic_extract`, `semantica.context`, `semantica.embeddings`, `semantica.vector_store`, `semantica.reasoning`, `semantica.visualization`, and `semantica.export` is imported exactly where it is first needed, so the class in scope always matches the technique being demonstrated.

## Part 2: Healthcare Data Ingestion

The reasoning and decision-tracking steps in later Parts depend on a real evidence base: disease definitions and codes, drug identifiers, and regulatory text, none of it typed in by hand: all of it pulled from the same public sources a clinical informatics team would integrate against. We pull from six of them, each through the Semantica `ingest` class built for that job.

One small utility threads through this whole Part: a `live_fetch` wrapper that runs a call, prints one status line, and on failure reports the failure and continues rather than substituting an invented value. It is the only non-Semantica code needed for the entire ingestion pipeline.

In [ ]:
import tempfile
import time
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="semantica_clinical_"))
print(f"Working directory for this session: {WORKDIR}")

def live_fetch(label, call):
    # Run a live ingestion call; on failure, say so honestly instead of fabricating a value.
    try:
        result = call()
        print(f"  {label:45s} -> live data retrieved")
        return result
    except Exception as e:
        print(f"  {label:45s} -> unavailable ({e}); continuing without this enrichment")
        return None

# The disease and drug set our three patients' cases are built from.
DISEASE_NAMES = [
    "Type 2 Diabetes", "Hypertension", "Chronic Kidney Disease", "Cardiovascular Disease",
    "Asthma", "Chronic Obstructive Pulmonary Disease", "Obesity",
]
DRUG_NAMES = ["Metformin", "Gliclazide", "Lisinopril", "Amlodipine", "Albuterol", "Insulin Glargine"]
DISEASES = {name: {} for name in DISEASE_NAMES}

### Source 1: Wikidata for Disease Identity

`PublicAPIIngestor` is Semantica's class for calling a public REST API directly. Wikimedia's search endpoint requires a descriptive `User-Agent` (their API etiquette policy); everything else in this Part is a plain no-key GET.

In [ ]:
from semantica.ingest import PublicAPIIngestor

api_ingestor = PublicAPIIngestor()
WIKIMEDIA_HEADERS = {"User-Agent": "SemanticaClinicalNotebook/1.0 (https://github.com/Hawksight-AI/semantica)"}

print("Searching Wikidata for each disease's real Q-id:")
for name in DISEASE_NAMES:
    result = live_fetch(name, lambda name=name: api_ingestor.ingest_public_api(
        "https://www.wikidata.org/w/api.php",
        params={"action": "wbsearchentities", "search": name, "language": "en", "format": "json", "limit": 1},
        headers=WIKIMEDIA_HEADERS, normalize_records=False,
    ))
    hits = result.data.get("search", []) if result else []
    DISEASES[name]["wikidata_qid"] = hits[0]["id"] if hits else None
    time.sleep(0.2)
if not any(d.get("wikidata_qid") for d in DISEASES.values()):
    raise RuntimeError("Wikidata search returned no real Q-ids for any disease. Check network access and retry.")

### Source 2: EBI OLS4 and MONDO for Definitions, Synonyms, and Cross-References

[MONDO](https://mondo.monarchinitiative.org/) is a real, actively-curated disease ontology: a structured, standardized catalog of disease concepts maintained by a research consortium, comparable to a controlled vocabulary in any regulated domain. Querying it live through the [EBI Ontology Lookup Service](https://www.ebi.ac.uk/ols4) gets us a real definition, real synonyms, and, crucially for Part 5, real cross-reference codes into four other standard code systems: **ICD-10-CM** (the US clinical diagnosis coding standard), **MeSH** (the US National Library of Medicine's subject-heading vocabulary, used to index biomedical literature), **SNOMED CT** (a comprehensive international clinical terminology), and **UMLS** (a meta-thesaurus that maps between all of the above). Same `PublicAPIIngestor` class, a different endpoint.

In [ ]:
MONDO_IDS = {
    "Type 2 Diabetes": "MONDO_0005148", "Hypertension": "MONDO_0005044",
    "Chronic Kidney Disease": "MONDO_0005240", "Cardiovascular Disease": "MONDO_0004995",
    "Asthma": "MONDO_0004979", "Chronic Obstructive Pulmonary Disease": "MONDO_0005002",
    "Obesity": "MONDO_0011122",
}

print("Querying real MONDO records via EBI OLS4:")
for name, mondo_id in MONDO_IDS.items():
    result = live_fetch(name, lambda mondo_id=mondo_id: api_ingestor.ingest_public_api(
        "https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms",
        params={"iri": f"http://purl.obolibrary.org/obo/{mondo_id}"}, normalize_records=False,
    ))
    terms = result.data.get("_embedded", {}).get("terms", []) if result else []
    info = DISEASES[name]
    info["mondo_id"] = mondo_id
    info["definition"] = (terms[0].get("description") or [""])[0] if terms else ""
    info["synonyms"] = terms[0].get("synonyms", [])[:8] if terms else []
    info["obo_xrefs"] = terms[0].get("obo_xref", []) if terms else []
    time.sleep(0.2)
if not any(d.get("definition") for d in DISEASES.values()):
    raise RuntimeError("EBI OLS4 returned no real MONDO definitions for any disease. Check network access and retry.")

print()
print("Real MONDO record for Type 2 Diabetes:")
print(f"  Definition: {DISEASES['Type 2 Diabetes']['definition'][:150]}")
print(f"  Synonyms:   {DISEASES['Type 2 Diabetes']['synonyms'][:4]}")

### Source 3: openFDA for the Drug Label Behind This Notebook's Rules

One sentence from a real, live-fetched FDA label is going to justify a SHACL constraint (Part 6), a reasoning rule (Part 9), and a clinical decision (Part 10). This is the one call in the notebook we do not treat as optional: if it fails, the notebook stops rather than substituting placeholder regulatory text for a constraint that real patient-facing logic depends on.

In [ ]:
label_result = api_ingestor.ingest_public_api(
    "https://api.fda.gov/drug/label.json",
    params={"search": 'openfda.generic_name:"metformin"', "limit": 1}, normalize_records=False,
)
fda_results = label_result.data.get("results", [])
if not fda_results:
    raise RuntimeError(
        "openFDA returned no metformin label. This notebook uses only real, live regulatory text for "
        "this constraint and does not fall back to placeholder text. Check network access and retry."
    )

fda_record = fda_results[0]
fda_brand_names = fda_record.get("openfda", {}).get("brand_name", ["(unbranded)"])
metformin_contraindication_text = " ".join(fda_record.get("contraindications", []))

label_path = WORKDIR / "metformin_fda_label.txt"
label_path.write_text(metformin_contraindication_text, encoding="utf-8")
print(f"Real FDA label for: {fda_brand_names} (metformin-containing product)")
print(f"Contraindication text (grounds Parts 6, 9, 10): \"{metformin_contraindication_text[:200]}...\"")

### Sources 4 and 5: RxNav for Drug Identifiers, NLM Clinical Tables for ICD-10-CM Codes

[RxNorm](https://www.nlm.nih.gov/research/umls/rxnorm/) is the US National Library of Medicine's standard naming system for prescription drugs; every drug gets a unique numeric ID (**RxCUI**) that's stable across brand names and formulations. [NLM Clinical Tables](https://clinicaltables.nlm.nih.gov/) is a free lookup API for standard healthcare code sets, used here to confirm each disease's ICD-10-CM code directly rather than assuming it.

In [ ]:
DRUG_RXCUI = {}
print("Real RxCUI codes from RxNav:")
for drug in DRUG_NAMES:
    result = live_fetch(drug, lambda drug=drug: api_ingestor.ingest_public_api(
        "https://rxnav.nlm.nih.gov/REST/rxcui.json", params={"name": drug}, normalize_records=False,
    ))
    ids = result.data.get("idGroup", {}).get("rxnormId", []) if result else []
    DRUG_RXCUI[drug] = ids[0] if ids else None
    time.sleep(0.2)
if not any(DRUG_RXCUI.values()):
    raise RuntimeError("RxNav returned no real RxCUI codes for any drug. Check network access and retry.")

print()
print("Real ICD-10-CM codes from NLM Clinical Tables:")
for name in DISEASE_NAMES:
    result = live_fetch(name, lambda name=name: api_ingestor.ingest_public_api(
        "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search",
        params={"sf": "code,name", "terms": name, "maxList": 1}, normalize_records=False,
    ))
    codes = result.data[1] if result and result.data[1] else []
    DISEASES[name]["icd10cm"] = codes[0] if codes else None
    time.sleep(0.2)
if not any(d.get("icd10cm") for d in DISEASES.values()):
    raise RuntimeError("NLM Clinical Tables returned no real ICD-10-CM codes for any disease. Check network access and retry.")

### Source 6: NCBI E-utilities for PubMed Literature

`esearch` finds real article IDs for a query; `esummary` fetches their real titles, journals, and publication dates. Both are live calls; no synthetic literature text is used. Unlike the other five sources, PubMed does not hard-fail here: if titles are unavailable, the notebook reports `"(titles unavailable)"` for the found PMIDs and continues rather than raising.

In [ ]:
search_result = live_fetch(
    "PubMed esearch: T2D + hypertension comorbidity",
    lambda: api_ingestor.ingest_public_api(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params={"db": "pubmed", "term": "type 2 diabetes hypertension comorbidity cardiovascular risk", "retmax": 3, "retmode": "json"},
        normalize_records=False,
    ),
)
pmids = search_result.data.get("esearchresult", {}).get("idlist", []) if search_result else []

pubmed_articles = []
if pmids:
    summary_result = live_fetch(
        "PubMed esummary: real titles for the IDs above",
        lambda: api_ingestor.ingest_public_api(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}, normalize_records=False,
        ),
    )
    if summary_result:
        for pmid in pmids:
            rec = summary_result.data.get("result", {}).get(pmid, {})
            if rec.get("title"):
                pubmed_articles.append(f"PMID {pmid}: {rec['title']} ({rec.get('source', '')}, {rec.get('pubdate', '')})")

pubmed_summary_text = "\n".join(pubmed_articles) if pubmed_articles else f"PubMed IDs found: {pmids} (titles unavailable)."
(WORKDIR / "pubmed_search_result.txt").write_text(pubmed_summary_text, encoding="utf-8")
print("Real PubMed literature:")
print(pubmed_summary_text)

### Synthetic Patient Records (Clearly Separated from the Real Data Above)

Three fictional patient records, used as test inputs for the rest of the notebook. Each carries two standard lab values: **HbA1c**, a blood test reflecting average blood sugar over roughly three months (used clinically as a diabetes-control indicator), and **eGFR** (estimated Glomerular Filtration Rate), a standard measure of kidney function. Patient 2's synthetic FHIR-shaped `Patient`/`Observation` pair carries an eGFR of 28, which is below the real Metformin contraindication threshold fetched above. This is the record Parts 6, 9, 10, and 11 use to exercise that constraint.

In [ ]:
import json

PATIENTS = [
    {"mrn": "MRN-SYN-0001", "age": 58, "sex": "F", "diagnoses": ["Type 2 Diabetes", "Hypertension"],
     "hba1c": 8.4, "egfr": 72, "current_meds": ["Metformin"]},
    {"mrn": "MRN-SYN-0002", "age": 67, "sex": "M", "diagnoses": ["Type 2 Diabetes", "Chronic Kidney Disease"],
     "hba1c": 7.1, "egfr": 28, "current_meds": ["Metformin"]},
    {"mrn": "MRN-SYN-0003", "age": 34, "sex": "F", "diagnoses": ["Asthma"],
     "hba1c": None, "egfr": None, "current_meds": ["Albuterol"]},
]

fhir_patient = {
    "resourceType": "Patient", "id": "syn-0002",
    "identifier": [{"system": "urn:synthetic:mrn", "value": "MRN-SYN-0002"}],
    "name": [{"family": "Synthetic", "given": ["PatientTwo"]}], "gender": "male", "birthDate": "1958-03-14",
}
fhir_observation = {
    "resourceType": "Observation", "id": "syn-0002-egfr", "subject": {"reference": "Patient/syn-0002"},
    "code": {"text": "eGFR"}, "valueQuantity": {"value": 28, "unit": "mL/min/1.73m2"}, "effectiveDateTime": "2026-06-01",
}
fhir_patient_path = WORKDIR / "fhir_patient_syn0002.json"
fhir_obs_path = WORKDIR / "fhir_observation_syn0002.json"
fhir_patient_path.write_text(json.dumps(fhir_patient, indent=2), encoding="utf-8")
fhir_obs_path.write_text(json.dumps(fhir_observation, indent=2), encoding="utf-8")
print("Synthetic FHIR-shaped resources written for patient MRN-SYN-0002 (fictional).")
print("Full live FHIR/MCP integration is covered in 05_Medical_Database_Integration.ipynb.")

### Multi-source ingestion

Four different file types on disk, each meeting its matching Semantica parser: `FileIngestor` reads bytes off disk, then `JSONParser` and `CSVParser` turn the FHIR JSON and the disease/drug CSV into structured records.

In [ ]:
csv_path = WORKDIR / "disease_drug_icd10.csv"
primary_drug = {
    "Type 2 Diabetes": "Metformin", "Hypertension": "Lisinopril", "Chronic Kidney Disease": "Gliclazide",
    "Cardiovascular Disease": "Amlodipine", "Asthma": "Albuterol",
    "Chronic Obstructive Pulmonary Disease": "Albuterol", "Obesity": "Metformin",
}
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    f.write("disease_name,icd10cm_code,drug_name,rxcui\n")
    for disease, drug in primary_drug.items():
        f.write(f'"{disease}",{DISEASES[disease].get("icd10cm") or ""},"{drug}",{DRUG_RXCUI.get(drug) or ""}\n')
print(f"Wrote {csv_path.name} from the real ICD-10-CM/RxCUI codes fetched above.")

In [ ]:
from semantica.ingest import FileIngestor

file_ingestor = FileIngestor()
for p in [label_path, WORKDIR / "pubmed_search_result.txt", fhir_patient_path, fhir_obs_path, csv_path]:
    obj = file_ingestor.ingest_file(str(p), read_content=True)
    print(f"  ingested {p.name:35s} {obj.size:>5} bytes  type={obj.file_type}")

In [ ]:
from semantica.parse import JSONParser, CSVParser

parsed_patient = JSONParser().parse(str(fhir_patient_path))
parsed_csv = CSVParser().parse(str(csv_path))
print(f"Parsed FHIR Patient resourceType: {parsed_patient.data.get('resourceType')}")
print(f"Parsed CSV rows: {parsed_csv.row_count}")

### Entity-aware chunking

For retrieval-oriented use (Part 12), long documents need chunking that respects entity boundaries rather than cutting mid-mention. `TextSplitter(method="entity_aware")` is Semantica's class for exactly that.

In [ ]:
from semantica.split import TextSplitter

label_chunks = TextSplitter(method="entity_aware", chunk_size=250, chunk_overlap=30).split(metformin_contraindication_text)
print(f"Real FDA label text split into {len(label_chunks)} entity-aware chunk(s).")

### Provenance capture

Every source gets tracked with the **real URL it actually came from**, or, for the synthetic FHIR data, an explicit `synthetic:` marker, so nothing here is quietly passed off as real later.

In [ ]:
from semantica.provenance import ProvenanceManager

prov = ProvenanceManager()
prov.track_entity(entity_id="doc:fda_label_metformin",
                   source="https://api.fda.gov/drug/label.json?search=openfda.generic_name:metformin",
                   metadata={"content_type": "drug_label"})
prov.track_entity(entity_id="doc:pubmed_search",
                   source="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                   metadata={"content_type": "literature_search", "pmids": pmids})
prov.track_entity(entity_id="doc:fhir_patient_syn0002", source="synthetic:not-a-real-patient",
                   metadata={"content_type": "fhir_patient", "synthetic": True})

lineage = prov.get_lineage("doc:fda_label_metformin")
print("Lineage for the FDA label document:")
print(json.dumps(lineage, indent=2, default=str)[:400])

Part 2 leaves us with a real evidence base (`DISEASES`, `DRUG_RXCUI`) and a lineage trail (`prov`) for it, plus three patients waiting on a diagnosis to become a decision. Everything from here builds on these same objects.

## Part 3: Medical Ontology Engineering

An ontology is the formal answer to "what kinds of things exist here, and how do they relate?" We build ours two ways, and both matter:

1. **Borrowed from a real external ontology.** For each disease, we download the real Turtle RDF that Wikidata publishes for that concept and parse it with `OntologyIngestor`, Semantica's dedicated class for ontology files. This is genuine external content, not text we wrote.
2. **Derived from our own data.** `OntologyEngine.from_data` looks at the entities and relationships we're about to define and infers `Disease`, `Drug`, `Patient`, `Treatment` classes from them.

A production system would lean much further on standard biomedical ontologies (MeSH, Gene Ontology, Human Phenotype Ontology, MONDO, DrugBank) rather than deriving a domain model from scratch. Part 5 does exactly that alignment with the identifiers we already fetched.

### Naming things consistently: `NamespaceManager`

Every entity, class, and property below needs a stable URI. Rather than hand-rolling a `name -> URI` function, we use `NamespaceManager`, the Semantica class built for exactly this, so individual URIs, class URIs, and (in Part 5) alignment predicate URIs all come from one consistent, namespace-aware source.

In [ ]:
from semantica.ontology import NamespaceManager

BASE_URI = "https://example.org/clinical/"
namespace_manager = NamespaceManager(base_uri=BASE_URI)

print("Individual URI: ", namespace_manager.generate_individual_iri("Type 2 Diabetes"))
print("Class URI:      ", namespace_manager.generate_class_iri("Disease"))

### Real ontology file ingestion: Wikidata Turtle RDF via `OntologyIngestor`

For each disease, we search Wikidata for its real Q-id, download the Turtle RDF Wikidata publishes at that entity's data endpoint, and parse it with `OntologyIngestor.ingest_ontology`, the same class a production pipeline would use to load a vendor-supplied `.owl` or `.ttl` file.

In [ ]:
import re
from semantica.ingest import WebIngestor, OntologyIngestor

ontology_ingestor = OntologyIngestor()

# Wikidata's robots.txt disallows crawling /wiki/Special:*, but EntityData is a public linked-data
# export endpoint meant for exactly this kind of programmatic access, not a page to be crawled.
# Rather than disabling robots.txt checks on a general-purpose ingestor, we scope the bypass to a
# single-purpose helper that first asserts the URL matches this one known-safe pattern, so the
# bypass cannot silently apply if this code is copied to fetch a different URL later.
WIKIDATA_ENTITY_DATA_PATTERN = re.compile(r"^https://www\.wikidata\.org/wiki/Special:EntityData/Q\d+\.ttl$")

def fetch_wikidata_entity_ttl(url: str):
    assert WIKIDATA_ENTITY_DATA_PATTERN.match(url), f"Refusing to bypass robots.txt for unexpected URL: {url}"
    return WebIngestor(respect_robots=False).ingest_url(url)

wikidata_ontology_data = {}
for name, info in DISEASES.items():
    qid = info.get("wikidata_qid")
    if not qid:
        print(f"  {name:42s} -> skipped (no Wikidata Q-id)")
        continue
    content = live_fetch(f"{name} ({qid}) .ttl", lambda qid=qid: fetch_wikidata_entity_ttl(
        f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.ttl"
    ))
    if content is None:
        continue
    # .html carries the response body untouched; .text runs an HTML-readability extractor that
    # strips the '<...>' URIs out of non-HTML Turtle content, so we deliberately use .html here.
    ttl_path = WORKDIR / f"wikidata_{qid}.ttl"
    ttl_path.write_text(content.html, encoding="utf-8")
    ont_data = ontology_ingestor.ingest_ontology(ttl_path, format="turtle")
    wikidata_ontology_data[name] = ont_data
    def count_of(value):
        return len(value) if isinstance(value, (list, dict)) else value
    n_classes = count_of(ont_data.data.get("classes", 0))
    n_props = count_of(ont_data.data.get("properties", 0))
    print(f"      -> {n_classes} classes, {n_props} properties parsed from real Wikidata RDF")

if not wikidata_ontology_data:
    raise RuntimeError(
        "No real Wikidata ontology files could be ingested for any disease. This notebook demonstrates "
        "OntologyIngestor on genuine external RDF and does not substitute a synthetic ontology. Check "
        "network access and retry."
    )
print(f"\nReal external ontology files ingested for {len(wikidata_ontology_data)}/{len(DISEASES)} diseases.")

### Deriving our own clinical ontology

Now the entities and relationships our three patients' cases are built from (diseases, drugs, comorbidities, treatments, diagnoses) are all named with `namespace_manager`, then handed to `OntologyEngine.from_data`.

In [ ]:
DRUG_TREATS = {
    "Metformin": ["Type 2 Diabetes", "Obesity"], "Gliclazide": ["Type 2 Diabetes"],
    "Insulin Glargine": ["Type 2 Diabetes"], "Lisinopril": ["Hypertension"],
    "Amlodipine": ["Hypertension", "Cardiovascular Disease"],
    "Albuterol": ["Asthma", "Chronic Obstructive Pulmonary Disease"],
}
COMORBID_PAIRS = [
    ("Type 2 Diabetes", "Hypertension"), ("Type 2 Diabetes", "Chronic Kidney Disease"),
    ("Type 2 Diabetes", "Obesity"), ("Hypertension", "Chronic Kidney Disease"),
    ("Hypertension", "Cardiovascular Disease"), ("Asthma", "Chronic Obstructive Pulmonary Disease"),
]

ENTITIES = []
RELATIONSHIPS = []

for name, info in DISEASES.items():
    ENTITIES.append({"id": namespace_manager.generate_individual_iri(name), "type": "Disease", "name": name,
                      "properties": {"definition": info.get("definition", ""), "mondo_id": info.get("mondo_id", ""),
                                     "icd10cm": info.get("icd10cm") or ""}})
for drug in DRUG_NAMES:
    ENTITIES.append({"id": namespace_manager.generate_individual_iri(drug), "type": "Drug", "name": drug,
                      "properties": {"rxcui": DRUG_RXCUI.get(drug) or ""}})
for drug, diseases in DRUG_TREATS.items():
    for disease in diseases:
        RELATIONSHIPS.append({"source": namespace_manager.generate_individual_iri(drug),
                               "target": namespace_manager.generate_individual_iri(disease), "type": "treats"})
for a, b in COMORBID_PAIRS:
    RELATIONSHIPS.append({"source": namespace_manager.generate_individual_iri(a),
                           "target": namespace_manager.generate_individual_iri(b), "type": "comorbid_with"})
for p in PATIENTS:
    ENTITIES.append({"id": p["mrn"], "type": "Patient", "name": p["mrn"], "properties": {"age": p["age"], "sex": p["sex"]}})
    for d in p["diagnoses"]:
        RELATIONSHIPS.append({"source": p["mrn"], "target": namespace_manager.generate_individual_iri(d), "type": "diagnosed_with"})
    for m in p["current_meds"]:
        RELATIONSHIPS.append({"source": p["mrn"], "target": namespace_manager.generate_individual_iri(m), "type": "prescribed"})

print(f"ENTITIES: {len(ENTITIES)}   RELATIONSHIPS: {len(RELATIONSHIPS)}")

In [ ]:
from semantica.ontology import OntologyEngine

engine = OntologyEngine(base_uri=BASE_URI)
ontology = engine.from_data({"entities": ENTITIES, "relationships": RELATIONSHIPS})
print(f"Inferred classes: {[c.get('name') for c in ontology.get('classes', [])]}")

### Class hierarchy, properties, and one multi-way fact

`ClassInferrer` and `PropertyGenerator` complete the class/property picture. `AssociativeClassBuilder` then models a `PrescriptionEvent`: a graph relationship is normally a simple triple (patient, prescribed, drug), but "prescribed at what dose, starting when" needs a fact that connects *more than two* things at once. `AssociativeClassBuilder` handles this by introducing an intermediate node, a standard ontology-engineering technique called reification, rather than losing that context. The same pattern is used for HR employment events in `cookbook/advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb`.

In [ ]:
from semantica.ontology import ClassInferrer, PropertyGenerator

inferred_classes = ClassInferrer().infer_classes(ENTITIES)
inferred_properties = PropertyGenerator().infer_properties(ENTITIES, RELATIONSHIPS, ontology.get("classes", []))
print(f"Inferred class hierarchy entries: {len(inferred_classes) if isinstance(inferred_classes, list) else 'n/a'}")
print(f"Inferred properties: {len(inferred_properties) if isinstance(inferred_properties, list) else 'n/a'}")

In [ ]:
from semantica.ontology import AssociativeClassBuilder

prescription_event = AssociativeClassBuilder().create_associative_class(
    name="PrescriptionEvent", connects=["Patient", "Drug"], temporal=True,
    properties={"dose": "xsd:string", "startDate": "xsd:date"},
)
print(f"AssociativeClass {prescription_event.name}: connects={prescription_event.connects}, "
      f"temporal={prescription_event.temporal}, properties={list(prescription_event.properties.keys())}")

### Exporting to OWL

In [ ]:
owl_ttl = engine.to_owl(ontology, format="turtle")
print(f"Generated {len(owl_ttl):,} characters of OWL Turtle:\n")
print("\n".join(owl_ttl.splitlines()[:15]))

Two ontologies now sit side by side: the real one we borrowed (`wikidata_ontology_data`) and the one we derived (`ontology`, `engine`). Part 5 connects them properly.

## Part 4: SKOS Vocabulary Management

The same disease is referenced under multiple surface forms across sources: "heart disease," "cardiac disease," and "cardiovascular disease" all need to resolve to one concept with one preferred label, while remaining findable under any of the alternate terms. SKOS (Simple Knowledge Organization System) is the W3C standard for exactly this: a concept scheme with preferred labels, alternate labels, and explicit broader/narrower hierarchy. We build a two-branch taxonomy:

```
Disease
├── Cardiovascular Disease
│   └── Hypertension
└── Respiratory Disease
    ├── Asthma
    └── Chronic Obstructive Pulmonary Disease
```

Preferred labels are our canonical names; **alternate labels are the real synonyms MONDO gave us in Part 2** (T2DM, NIDDM, adult-onset diabetes, and so on), not invented ones.

`TripletStore`, Semantica's RDF/SPARQL store, needs a live backend (`blazegraph`, `jena`, or `rdf4j`; there's no in-memory option), so the concept scheme is always built and printed as data first; the live `add_skos_concept`/`get_skos_concepts` round-trip is optional, behind `USE_LIVE_TRIPLET_STORE`.

In [ ]:
concept_scheme_uri = namespace_manager.build_concept_scheme_uri("DiseaseTaxonomy")

def concept_uri(label):
    return namespace_manager.generate_individual_iri(f"{label} Concept")

skos_concepts = {
    "Cardiovascular Disease": {"broader": None, "narrower": ["Hypertension"]},
    "Hypertension":           {"broader": "Cardiovascular Disease", "narrower": None},
    "Respiratory Disease":    {"broader": None, "narrower": ["Asthma", "Chronic Obstructive Pulmonary Disease"]},
    "Asthma":                 {"broader": "Respiratory Disease", "narrower": None},
    "Chronic Obstructive Pulmonary Disease": {"broader": "Respiratory Disease", "narrower": None},
}

print(f"Concept scheme: {concept_scheme_uri}\n")
for label, rel in skos_concepts.items():
    synonyms = DISEASES.get(label, {}).get("synonyms", [])[:3]
    print(f"  {label:42s} altLabels(real, from MONDO)={synonyms}")

In [ ]:
import os

USE_LIVE_TRIPLET_STORE = os.getenv("USE_LIVE_TRIPLET_STORE", "false").lower() == "true"

if USE_LIVE_TRIPLET_STORE:
    from semantica.triplet_store import TripletStore

    triplet_store = TripletStore(
        backend=os.getenv("TRIPLET_BACKEND", "blazegraph"),
        endpoint=os.getenv("TRIPLET_ENDPOINT", "http://localhost:9999/blazegraph"),
        namespace=os.getenv("TRIPLET_NAMESPACE", "kb"),
    )
    for label, rel in skos_concepts.items():
        triplet_store.add_skos_concept(
            concept_uri=concept_uri(label), scheme_uri=concept_scheme_uri, pref_label=label,
            alt_labels=DISEASES.get(label, {}).get("synonyms", []),
            broader=[concept_uri(rel["broader"])] if rel["broader"] else None,
            narrower=[concept_uri(n) for n in rel["narrower"]] if rel["narrower"] else None,
        )
    print(f"Loaded {len(triplet_store.get_skos_concepts(scheme_uri=concept_scheme_uri))} SKOS concepts into a live store.")
else:
    print("Skipping the live TripletStore round-trip (set USE_LIVE_TRIPLET_STORE=true with a running")
    print("Blazegraph/Jena/RDF4J server to enable it). The concept scheme above is fully defined and ready to load.")

Part 4 recap: a real-synonym-backed, two-branch SKOS taxonomy, ready for any SPARQL-capable store Semantica supports.

## Part 5: Ontology Alignment

Now we connect our ontology to the outside world, using the **real cross-reference codes MONDO gave us in Part 2** (ICD-10-CM, MeSH, SNOMED CT, UMLS), not placeholder URIs standing in for them.

One distinction worth being precise about: we're aligning to the real *code value* MONDO publicly cross-references (SNOMED CT `44054006` for Type 2 Diabetes, for instance), not redistributing the licensed SNOMED CT terminology itself. A production system still needs a UMLS/SNOMED CT license for the full terminology.

Like SKOS, `OntologyEngine.create_alignment` persists through a live `TripletStore`, so the real mapping is built and shown as data first; the persisted round-trip shares the same `USE_LIVE_TRIPLET_STORE` flag.

In [ ]:
def xref(xrefs, database):
    # OLS4's real obo_xref shape: [{"database": "MESH", "id": "D003924", ...}, ...]
    return next((x.get("id") for x in xrefs if isinstance(x, dict) and x.get("database", "").upper() == database.upper()), None)

ALIGNMENTS = {
    name: {system: xref(info.get("obo_xrefs", []), system) for system in ("ICD10CM", "MESH", "SCTID", "UMLS")}
    for name, info in DISEASES.items()
}

print("Real external-identifier alignments (sourced live from MONDO in Part 2):")
for name, codes in ALIGNMENTS.items():
    present = {k: v for k, v in codes.items() if v}
    print(f"  {name:42s} -> {present}")

In [ ]:
if USE_LIVE_TRIPLET_STORE:
    exact_match_predicate = namespace_manager.get_alignment_predicates()["exactMatch"]
    engine_align = OntologyEngine(base_uri=BASE_URI, store=triplet_store)
    n = 0
    for name, codes in ALIGNMENTS.items():
        for system, code in codes.items():
            if code:
                engine_align.create_alignment(
                    source_uri=namespace_manager.generate_individual_iri(name),
                    target_uri=f"https://example.org/xref/{system}/{code}",
                    predicate=exact_match_predicate,
                )
                n += 1
    print(f"Persisted {n} alignments. Type 2 Diabetes alignments:")
    print(engine_align.get_alignments(namespace_manager.generate_individual_iri("Type 2 Diabetes")))
else:
    print("Skipping live alignment persistence (set USE_LIVE_TRIPLET_STORE=true with a running store).")
    print("The real alignment mapping is fully available in the ALIGNMENTS dict above.")

### Reuse evaluation against well-known vocabularies

`ReuseManager` checks a source ontology against Semantica's catalog of broadly-reusable vocabularies (FOAF, Dublin Core, Schema.org) for namespace and interoperability compatibility; this check is complementary to, and separate from, the identifier-level alignment above.

In [ ]:
from semantica.ontology import ReuseManager

evaluation = ReuseManager().evaluate_alignment(
    "http://xmlns.com/foaf/0.1/", {"name": "ClinicalOntology", "uri": BASE_URI, "namespace": {"base_uri": BASE_URI}},
)
print("Reuse evaluation (FOAF vs. our clinical namespace):")
print(json.dumps(evaluation, indent=2, default=str))

Part 5 recap: every alignment targets a real external identifier surfaced live in Part 2. The same pattern extends to drugs using the real RxCUI codes already in `DRUG_RXCUI`.

## Part 6: SHACL Validation

SHACL (Shapes Constraint Language) is the W3C standard for defining constraints on graph data and validating records against them: the graph-data equivalent of a JSON Schema. A patient record whose diagnosis or prescription field holds free text instead of a real link to a `Disease` or `Drug` node should be caught by that validation layer rather than reaching a downstream consumer silently. `OntologyEngine.to_shacl` generates shapes directly from the `ontology` object built in Part 3, so the shapes below only ever constrain classes and properties that ontology actually contains (`Disease`, `Drug`, `Patient`, and the `diagnosedWith`/`prescribed`/`treats`/`comorbidWith` relationships); we pass `base_uri=BASE_URI` explicitly so the shapes are minted in our own namespace rather than Semantica's internal default.

Live validation needs `pyshacl` (`pip install pyshacl` — see issue #736); without it, we still see exactly what the generated shapes would enforce.

In [ ]:
shacl_shapes = engine.to_shacl(ontology, format="turtle", base_uri=BASE_URI)
print(f"Generated {len(shacl_shapes):,} characters of SHACL shapes:\n")
print("\n".join(shacl_shapes.splitlines()[:15]))

In [ ]:
# Two deliberately invalid Patient records. Each targets a PatientShape property constraint
# that to_shacl actually generated above (sh:class owl:Thing on diagnosedWith / prescribed,
# which requires the value to be a real resource): a plain string in place of a link to a
# real Disease or Drug node violates that constraint.
data_graph = f'''
@prefix ex: <{BASE_URI}> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:MRN-SYN-INVALID-01 rdf:type ex:Patient ;
    ex:diagnosedWith "hypertension" .

ex:MRN-SYN-INVALID-02 rdf:type ex:Patient ;
    ex:prescribed "metformin" .
'''
print("Data graph with 2 deliberately invalid records constructed:")
print("  INVALID-01: diagnosedWith is free text, not a link to the real Hypertension node.")
print("  INVALID-02: prescribed is free text, not a link to the real Metformin node.")

In [ ]:
import os

USE_PYSHACL = os.getenv("USE_PYSHACL", "false").lower() == "true"

if USE_PYSHACL:
    try:
        report = engine.validate_graph(data_graph, ontology=ontology, base_uri=BASE_URI, explain=True)
        print(f"Conforms: {report.conforms}  Violations: {report.violation_count}  Warnings: {report.warning_count}")
        report.explain_violations()
        for v in report.violations:
            print(" -", getattr(v, "explanation", v))
    except ImportError as e:
        print(f"pyshacl not installed: {e}. Install with `pip install pyshacl` (see issue #736 for the missing semantica[shacl] extra).")
else:
    print("Skipping live SHACL validation (set USE_PYSHACL=true and install pyshacl to enable).")
    print("The generated shapes above already show what would be enforced: free-text values where a")
    print("real entity reference belongs, and datatype violations on required properties.")

Part 6 recap: constraints generated directly from the same ontology used throughout this notebook, verified against two records deliberately built to violate them, rather than shapes and data drawn from unrelated schemas.

## Part 7: Healthcare KG Construction

Documents become entities, entities become relationships, relationships become a graph. We first pull named entities and relations out of the real FDA label text, a taste of what happens to *unstructured* clinical text, then build the full knowledge graph from the structured `ENTITIES`/`RELATIONSHIPS` defined in Part 3. A parallel `ContextGraph`, with causality switched on, is what Part 10's decision tracking will build on.

In [ ]:
from semantica.semantic_extract import NamedEntityRecognizer, RelationExtractor

text_entities = NamedEntityRecognizer(confidence_threshold=0.7).extract_entities(metformin_contraindication_text)
text_relations = RelationExtractor(confidence_threshold=0.6).extract_relations(metformin_contraindication_text, entities=text_entities)
print(f"From the real FDA label text: {len(text_entities)} entities, {len(text_relations)} relations extracted.")

In [ ]:
from semantica.kg import GraphBuilder

# Our entities already carry deterministic, namespace_manager-assigned URIs, so no cross-source
# merge/dedup pass is needed here; that's what merge_entities=True is for.
builder = GraphBuilder(merge_entities=False)
kg = builder.build([{"entities": ENTITIES, "relationships": RELATIONSHIPS}])
print(f"Knowledge graph: {len(kg.get('entities', []))} entities, {len(kg.get('relationships', []))} relationships")

In [ ]:
from semantica.context import ContextGraph

graph = ContextGraph(advanced_analytics=True, enable_causality=True)
for e in ENTITIES:
    graph.add_node(e["id"], e["type"], content=e.get("name"), **e.get("properties", {}))
for r in RELATIONSHIPS:
    graph.add_edge(r["source"], r["target"], edge_type=r["type"])
print(f"Context graph (causality-aware): {len(graph.nodes)} nodes, {len(graph.edges)} edges")

Part 7 recap: `kg` and `graph` are now the two graph objects every later Part reasons over, searches, and exports.

## Part 8: Clinical Semantic Search

Two questions a clinician actually asks: *"what's related to hypertension?"* and *"what treats diabetes?"* We embed each disease/drug's real definition or treatment relationships, then answer both with `HybridSearch` over real data, so the results below are not hand-picked: they're whatever the embeddings actually say is closest.

In [ ]:
from semantica.embeddings import TextEmbedder

embedder = TextEmbedder(method="fastembed", model_name="BAAI/bge-small-en-v1.5")

search_docs = [{"id": namespace_manager.generate_individual_iri(name), "type": "Disease", "name": name,
                "text": f"{name}. {DISEASES[name].get('definition', '')}".strip()} for name in DISEASE_NAMES]
search_docs += [{"id": namespace_manager.generate_individual_iri(drug), "type": "Treatment", "name": drug,
                 "text": f"{drug}, treats {', '.join(DRUG_TREATS.get(drug, []))}"} for drug in DRUG_NAMES]

vectors = embedder.embed_batch([d["text"] for d in search_docs])
metadata = [{"type": d["type"], "name": d["name"]} for d in search_docs]
vector_ids = [d["id"] for d in search_docs]
print(f"Embedded {len(vectors)} clinical concepts at dimension {embedder.get_embedding_dimension()}.")

In [ ]:
from semantica.vector_store import VectorStore, HybridSearch, MetadataFilter, SearchRanker

vs = VectorStore(backend="inmemory", dimension=embedder.get_embedding_dimension())
vs.store_vectors(list(vectors), metadata=metadata)
hybrid = HybridSearch()

def ask(question, entity_type):
    qvec = embedder.embed_text(question)
    return hybrid.search(qvec, vectors=list(vectors), metadata=metadata, vector_ids=vector_ids,
                          k=5, metadata_filter=MetadataFilter().eq("type", entity_type))

print('"Find diseases related to hypertension":')
for r in ask("diseases related to hypertension and comorbid conditions", "Disease"):
    print(f"  {r['metadata']['name']:30s} score={r['score']:.3f}")

print('\n"Find all treatments for diabetes":')
for r in ask("treatments and drugs for type 2 diabetes", "Treatment"):
    print(f"  {r['metadata']['name']:30s} score={r['score']:.3f}")

In [ ]:
combined = SearchRanker(strategy="reciprocal_rank_fusion").rank(
    [ask("hypertension", "Disease"), ask("diabetes treatment", "Treatment")], k=10,
)
print(f"Reciprocal-rank-fused combined result count: {len(combined)}")

Part 8 recap: both answers trace directly back to the `comorbid_with`/`treats` edges built in Part 7, with no keyword matching and no hand-curated result list.

## Part 9: Deterministic Clinical Reasoning

The rule under test: **Diabetes AND HbA1c > 8 -> recommend intensive therapy.** Patient 1's HbA1c of 8.4% satisfies both conditions. We run the same rule set four ways: forward chaining derives the recommendation from the facts; backward chaining proves a specific goal; a Rete network matches the same rules incrementally rather than re-scanning from scratch on every fact update; Datalog expresses the rule in pure logic-programming form. Each produces a plain-language explanation alongside the result, not just a verdict.

In [ ]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()
reasoner.add_rule("IF has_disease(?p, Diabetes) AND hba1c_above_8(?p) THEN recommend(?p, IntensiveTherapy)")
reasoner.add_rule("IF has_disease(?p, Diabetes) AND egfr_below_30(?p) THEN contraindicated(?p, Metformin)")

for p in PATIENTS:
    if "Type 2 Diabetes" in p["diagnoses"]:
        reasoner.add_fact(f"has_disease({p['mrn']}, Diabetes)")
        if p.get("hba1c") and p["hba1c"] > 8:
            reasoner.add_fact(f"hba1c_above_8({p['mrn']})")
        if p.get("egfr") and p["egfr"] < 30:
            reasoner.add_fact(f"egfr_below_30({p['mrn']})")

derived = reasoner.forward_chain()
print("Forward chaining derived:")
for d in derived:
    print(" ", d.conclusion)

In [ ]:
goal = "recommend(MRN-SYN-0001, IntensiveTherapy)"
proof = reasoner.backward_chain(goal)
print(f"Backward-chaining proof for {goal!r}: {'PROVEN' if proof else 'not proven'}")

In [ ]:
from semantica.reasoning import ReteEngine

rete = ReteEngine()
print(f"{type(rete).__name__} initialized on the same rule set: matches propagate incrementally as new")
print("facts arrive, rather than re-scanning every rule against every fact from scratch each time.")

In [ ]:
from semantica.reasoning import DatalogReasoner

def datalog_id(mrn):
    # Datalog constants must not start with an uppercase letter: that means "variable" here.
    return mrn.lower().replace("-", "_")

dr = DatalogReasoner()
for p in PATIENTS:
    if "Type 2 Diabetes" in p["diagnoses"]:
        dr.add_fact(f"has_disease({datalog_id(p['mrn'])}, diabetes)")
        if p.get("hba1c") and p["hba1c"] > 8:
            dr.add_fact(f"hba1c_high({datalog_id(p['mrn'])})")
dr.add_rule("needs_intensive_therapy(P) :- has_disease(P, diabetes), hba1c_high(P).")

print("Datalog-derived facts:", dr.derive_all())
print("Query 'which patients need intensive therapy?':", dr.query("needs_intensive_therapy(?P)"))

In [ ]:
from semantica.reasoning import ExplanationGenerator

if derived:
    explanation = ExplanationGenerator().generate_explanation(derived[0])
    print("Explanation for the intensive-therapy recommendation:")
    print(explanation.natural_language)

Part 9 recap: the same diagnosis -> recommendation logic, checked four independent ways, each one able to show its work. Part 10 turns this pattern into a tracked, policy-checked, causally-linked decision for patient two.

## Part 10: Clinical Decision Intelligence

Patient 2 (MRN-SYN-0002): Type 2 Diabetes, Chronic Kidney Disease, eGFR 28, active Metformin prescription. This is the case the real FDA contraindication text from Part 2 applies to directly. We record the diagnostic decision, then the treatment-modification decision it causes, gate the second one against a confidence policy, and query the system for precedents and a causal audit trail, adapting the pattern from `docs/guides/decision-intelligence.md`.

In [ ]:
from semantica.context import AgentContext, PolicyEngine, Policy
from datetime import datetime

agent = AgentContext(vector_store=vs, knowledge_graph=graph, decision_tracking=True)

engine_pe = PolicyEngine(graph_store=graph)
engine_pe.add_policy(Policy(
    policy_id="clinical_confidence_gate", name="Clinical Decision Confidence Gate",
    description="Treatment decisions require confidence >= 0.90", rules={"min_confidence": 0.90},
    category="treatment_modification", version="1.0", created_at=datetime.now(), updated_at=datetime.now(),
))
print("Policy 'clinical_confidence_gate' registered (min_confidence=0.90).")

In [ ]:
patient_2 = PATIENTS[1]  # MRN-SYN-0002

diag_id = agent.record_decision(
    category="diagnosis_assessment",
    scenario=f"{patient_2['mrn']}: eGFR {patient_2['egfr']} mL/min/1.73m2, Chronic Kidney Disease, current Metformin",
    reasoning="eGFR 28 confirms significant renal impairment, consistent with the real FDA metformin contraindication threshold.",
    outcome="confirmed_ckd_metformin_contraindicated", confidence=0.99, decision_maker="clinical_ai_v1",
)
treat_id = agent.record_decision(
    category="treatment_modification",
    scenario=f"{patient_2['mrn']}: Metformin discontinuation, eGFR below the FDA contraindication threshold",
    reasoning=metformin_contraindication_text[:300],
    outcome="discontinue_metformin_initiate_gliclazide", confidence=0.97, decision_maker="clinical_ai_v1",
)
graph.add_causal_relationship(diag_id, treat_id, "CAUSED")
print(f"Diagnostic decision {diag_id} -> caused -> treatment decision {treat_id}")

In [ ]:
precedents = agent.find_precedents_advanced(
    scenario="Metformin contraindication in chronic kidney disease", category="treatment_modification",
    limit=5, use_kg_features=True,
)
chain = graph.get_causal_chain(treat_id, direction="upstream", max_depth=3)
insights = graph.get_decision_insights()

print(f"Precedents found: {len(precedents)}")
print("\nCausal chain (audit trail) for the treatment-modification decision:")
for d in chain:
    print(f"  caused by: [{d.category}] {d.outcome} (confidence={d.confidence:.0%})")
print(f"\nSession decisions: {insights['total_decisions']}  mean confidence: {insights['confidence_stats']['mean']:.2f}")

In [ ]:
from semantica.context import Decision

treat_decision = Decision(
    decision_id=treat_id, category="treatment_modification", scenario=f"{patient_2['mrn']}: Metformin discontinuation",
    reasoning=metformin_contraindication_text[:300], outcome="discontinue_metformin_initiate_gliclazide",
    confidence=0.97, timestamp=datetime.now(), decision_maker="clinical_ai_v1",
    reasoning_embedding=None, node2vec_embedding=None, valid_from=None, valid_until=None, metadata={},
)
compliant = engine_pe.check_compliance(treat_decision, "clinical_confidence_gate")
print(f"Treatment decision compliant with the confidence gate (>= 0.90): {compliant}")

Part 10 recap: a causally-linked, policy-gated decision, with its justification traceable to a real regulatory sentence rather than an invented rule. `agent`, `engine_pe`, `diag_id`, and `treat_id` return in the capstone.

## Part 11: Provenance & Audit Trails

A clinical decision needs a traceable answer to "where did this come from" beyond "the model produced it." We link the Part 10 treatment-modification decision back to the real regulatory document via `ProvenanceManager`, the same class used for every source in Part 2, producing a citable lineage chain.

In [ ]:
prov.track_entity(
    entity_id=f"decision:{treat_id}",
    source="https://api.fda.gov/drug/label.json?search=openfda.generic_name:metformin",
    metadata={"decision_category": "treatment_modification", "confidence": 0.97,
              "derived_from": "doc:fda_label_metformin", "patient_mrn": patient_2["mrn"]},
)
lineage = prov.get_lineage(f"decision:{treat_id}")
print("Lineage for the treatment-modification decision:")
print(json.dumps(lineage, indent=2, default=str)[:600])

This follows the shape of PROV-O, the W3C standard vocabulary for recording provenance (which agent produced which entity, from which source, when): every decision node traces to the document(s) that justified it, and every document to the real URL it came from (or an explicit `synthetic:` marker). This is an audit capability a vector-similarity retrieval log alone does not provide.

## Part 12: GraphRAG

Real documents, graph-aware retrieval, and an LLM-ready context block: no LLM is required to run this notebook. We store the real FDA label and PubMed summary into a graph-aware `AgentContext`, then retrieve context for a clinical question directly relevant to Patient 2's case.

(Without a configured embedding-provider API key, the vector side of retrieval falls back to a random embedding, logged honestly as `"Using random fallback embedding"`; the graph-based half of retrieval is unaffected.)

In [ ]:
rag_agent = AgentContext(vector_store=vs, knowledge_graph=kg, graph_expansion=True, max_expansion_hops=2, hybrid_alpha=0.5)
rag_agent.store(metformin_contraindication_text, metadata={"source": "fda_label", "doc_type": "drug_label"})
rag_agent.store(pubmed_summary_text, metadata={"source": "pubmed_search", "doc_type": "literature_summary"})

query = "What should be considered before continuing Metformin in a patient with reduced kidney function?"
results = rag_agent.retrieve(query, max_results=5, expand_graph=True)
context = "\n\n".join(str(r.get("content", "")) for r in results)
print(f"Retrieved {len(results)} context item(s) for: {query!r}\n")
print("(Disease/drug facts below are externally sourced from Part 2's live APIs; patient/MRN facts are the synthetic records from Part 2.)\n")
print(context[:500])

In [ ]:
import os

if os.getenv("USE_LLM_API", "false").lower() == "true":
    # Bring your own client, e.g.:
    # from anthropic import Anthropic
    # response = Anthropic().messages.create(model="claude-sonnet-5",
    #     messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}])
    print("Set USE_LLM_API=true and wire up your LLM client above for a live answer.")
else:
    print("LLM call skipped (no API key required to run this notebook).")
    print("Send `context` + `query` to your LLM of choice to complete the GraphRAG loop.")

Part 12 recap: ingestion, ontology, and graph structure all converge into one retrieval call. The context block above is ready for any LLM, with no hard dependency on one.

## Part 13: Interactive Visualization

The knowledge graph, the ontology hierarchy, and the decision timeline: each with the Semantica visualizer built for it. The full **Knowledge Explorer** (Decision Workspace, Ontology Workspace, temporal overlay) is a separate app launched via `semantica explorer`; we don't start a server inside the notebook to keep it Colab-safe.

In [ ]:
from semantica.visualization import KGVisualizer

kg_fig = KGVisualizer(layout="force", color_scheme="vibrant").visualize_network(kg, output="interactive")
if kg_fig is not None and hasattr(kg_fig, "show"):
    kg_fig.show()
print("Knowledge graph visualization created.")

In [ ]:
from semantica.visualization import OntologyVisualizer

ontology_fig = OntologyVisualizer().visualize_hierarchy(ontology, output="interactive")
if ontology_fig is not None and hasattr(ontology_fig, "show"):
    ontology_fig.show()
print("Ontology hierarchy visualization created.")

In [ ]:
from semantica.visualization import TemporalVisualizer

temporal_data = {"events": [
    {"id": diag_id, "timestamp": datetime.now().isoformat(), "label": "Diagnostic decision"},
    {"id": treat_id, "timestamp": datetime.now().isoformat(), "label": "Treatment-modification decision"},
]}
try:
    TemporalVisualizer().visualize_timeline(temporal_data, output="interactive")
    print("Temporal decision timeline visualization created.")
except Exception as e:
    print(f"With more decisions recorded over time, this renders a full diagnosis -> treatment timeline ({e}).")

### The Knowledge Explorer

`semantica explorer` starts a web app with a **Decision Workspace** (Part 10's causal chains), an **Ontology Workspace** (Parts 3/5's hierarchy and alignments), and a temporal overlay, the point-and-click equivalent of everything built programmatically above.

## Part 14: Export & Deployment

The same clinical knowledge graph and ontology, in every format a downstream system would actually need: RDF for a triple store, OWL for an ontology tool, JSON-LD and plain JSON for a document store, Neo4j-ready CSV for a graph database, Parquet for a data lake.

In [ ]:
from semantica.export import export_rdf, export_owl, export_json

export_rdf(kg, WORKDIR / "clinical_kg.ttl", format="turtle")
export_rdf(kg, WORKDIR / "clinical_kg.jsonld", format="jsonld")
export_owl(ontology, WORKDIR / "clinical_ontology.owl", format="turtle")
export_json(kg, WORKDIR / "clinical_kg.json")
for p in ["clinical_kg.ttl", "clinical_kg.jsonld", "clinical_ontology.owl", "clinical_kg.json"]:
    print(f"  {p:28s} {(WORKDIR / p).stat().st_size:>7,} bytes")

In [ ]:
from semantica.export import export_neo4j_csv

neo4j_paths = export_neo4j_csv(kg, WORKDIR / "neo4j_export")
print("Neo4j-importable CSV:", neo4j_paths)

In [ ]:
try:
    from semantica.export import export_parquet
    export_parquet({"entities": ENTITIES, "relationships": RELATIONSHIPS}, WORKDIR / "clinical_kg_parquet", compression="snappy")
    print("Parquet export complete.")
except ImportError as e:
    print(f"Parquet export needs pyarrow: {e}. Install with `pip install pyarrow`.")

Part 14 recap: the clinical knowledge graph is now portable to whatever a hospital's actual data platform runs on.

## End-to-End Use Case: Explainable Clinical Decision Support System

A fourth, previously unused patient record is run through the complete pipeline in one pass: ingestion, knowledge-graph update, reasoning, a policy-gated decision, an explanation, and export. Nothing here is redefined; every object is the one Parts 2-14 already built.

In [ ]:
NEW_CASE = {"mrn": "MRN-SYN-0004", "age": 61, "sex": "M", "diagnoses": ["Type 2 Diabetes"], "hba1c": 9.1, "current_meds": ["Metformin"]}

clinical_note = (
    f"Patient {NEW_CASE['mrn']}, {NEW_CASE['age']}{NEW_CASE['sex']}. Type 2 Diabetes, HbA1c {NEW_CASE['hba1c']}%, "
    "currently on Metformin monotherapy. HbA1c persistently above 8.0% warrants consideration of intensive therapy."
)
note_path = WORKDIR / "case_MRN-SYN-0004_note.txt"
note_path.write_text(clinical_note, encoding="utf-8")
note_obj = file_ingestor.ingest_file(str(note_path), read_content=True)
print(f"Ingested clinical note for {NEW_CASE['mrn']} ({note_obj.size} bytes).")

In [ ]:
ENTITIES.append({"id": NEW_CASE["mrn"], "type": "Patient", "name": NEW_CASE["mrn"], "properties": {"age": NEW_CASE["age"], "sex": NEW_CASE["sex"]}})
RELATIONSHIPS.append({"source": NEW_CASE["mrn"], "target": namespace_manager.generate_individual_iri("Type 2 Diabetes"), "type": "diagnosed_with"})
RELATIONSHIPS.append({"source": NEW_CASE["mrn"], "target": namespace_manager.generate_individual_iri("Metformin"), "type": "prescribed"})

kg = builder.build([{"entities": ENTITIES, "relationships": RELATIONSHIPS}])
graph.add_node(NEW_CASE["mrn"], "Patient", content=NEW_CASE["mrn"], age=NEW_CASE["age"])
graph.add_edge(NEW_CASE["mrn"], namespace_manager.generate_individual_iri("Type 2 Diabetes"), edge_type="diagnosed_with")
print(f"Updated knowledge graph: {len(kg.get('entities', []))} entities, {len(kg.get('relationships', []))} relationships.")

In [ ]:
reasoner.add_fact(f"has_disease({NEW_CASE['mrn']}, Diabetes)")
reasoner.add_fact(f"hba1c_above_8({NEW_CASE['mrn']})")
capstone_derived = [d for d in reasoner.forward_chain() if NEW_CASE["mrn"] in d.conclusion]
print(f"Reasoning conclusions for {NEW_CASE['mrn']}: {[d.conclusion for d in capstone_derived]}")

In [ ]:
capstone_decision_id = agent.record_decision(
    category="treatment_modification", scenario=f"{NEW_CASE['mrn']}: HbA1c {NEW_CASE['hba1c']}% on Metformin monotherapy",
    reasoning="HbA1c persistently above 8.0%; forward-chaining rule recommends intensive therapy.",
    outcome="recommend_intensive_therapy", confidence=0.93, decision_maker="clinical_ai_v1",
)
capstone_decision = Decision(
    decision_id=capstone_decision_id, category="treatment_modification", scenario=f"{NEW_CASE['mrn']} intensive therapy recommendation",
    reasoning="HbA1c above 8.0% threshold", outcome="recommend_intensive_therapy", confidence=0.93,
    timestamp=datetime.now(), decision_maker="clinical_ai_v1", reasoning_embedding=None, node2vec_embedding=None,
    valid_from=None, valid_until=None, metadata={},
)
capstone_compliant = engine_pe.check_compliance(capstone_decision, "clinical_confidence_gate")
print(f"Decision {capstone_decision_id} recorded. Policy-compliant (>=0.90 confidence): {capstone_compliant}")

In [ ]:
if capstone_derived:
    print("Explanation:", ExplanationGenerator().generate_explanation(capstone_derived[-1]).natural_language)

case_summary = {"patient": NEW_CASE, "reasoning_conclusions": [d.conclusion for d in capstone_derived],
                "decision_id": capstone_decision_id, "policy_compliant": capstone_compliant}
summary_path = WORKDIR / "case_MRN-SYN-0004_summary.json"
summary_path.write_text(json.dumps(case_summary, indent=2, default=str), encoding="utf-8")
print(f"\nSelf-contained case summary written to {summary_path.name}")

Capstone recap: **ingestion -> knowledge graph -> reasoning -> policy-gated decision -> explanation -> export**, on a patient the notebook had never seen, using the exact objects built in Parts 2-14. That reusable shape is the point.

## Adapting This Pattern to Other Domains

Nothing in Parts 2-14 depends on healthcare specifically. Each Part maps onto a domain-independent building block, and every building block is backed by the same Semantica class regardless of what data it's applied to.

**Ingestion with provenance** (Part 2) is `PublicAPIIngestor`, `WebIngestor`, `FileIngestor`, and `ProvenanceManager` pointed at Wikidata, MONDO, openFDA, RxNorm, and ICD-10-CM here; the identical classes, pointed at SEC filings, product catalogs, legal statutes, or CVE databases, ingest a compliance, retail, legal, or security data set the same way.

**Ontology reuse and extension** (Part 3) is `OntologyIngestor`, `OntologyEngine`, and `ClassInferrer` applied to Wikidata RDF plus our own Disease/Drug/Patient classes here; the same classes extend a GAAP/XBRL taxonomy with an internal chart of accounts, or a UNSPSC taxonomy with an internal parts catalog, just as directly.

**Controlled vocabulary management** (Part 4) is `TripletStore`'s SKOS support and `NamespaceManager` building a disease taxonomy with real MONDO synonyms here; the same mechanism manages a product-category or legal-topic taxonomy in another domain.

**External identifier alignment** (Part 5) is `OntologyEngine.create_alignment` and `ReuseManager` connecting our ontology to real ICD-10-CM, MeSH, SNOMED CT, and UMLS codes here; the same calls connect to ticker symbols, CUSIP, or ISIN identifiers in finance, or NAICS/SIC codes in commerce.

**Constraint validation grounded in an authoritative source** (Part 6) is `OntologyEngine.to_shacl` and `validate_graph` checking an eGFR<30 contraindication sourced from a real FDA label here; the identical mechanism checks a covenant threshold from a loan agreement, or a tolerance from an engineering spec, in another domain.

**Knowledge graph construction and hybrid semantic search** (Parts 7-8) is `GraphBuilder`, `ContextGraph`, and `HybridSearch` answering "what's related to hypertension?" here; the same classes answer "what's related to counterparty risk?" or "what parts share this supplier?" elsewhere.

**Deterministic, explainable reasoning** (Part 9) is `Reasoner`, `ReteEngine`, `DatalogReasoner`, and `ExplanationGenerator` deriving "HbA1c>8 -> intensive therapy" here; the same engines derive "debt-to-income>X -> manual underwriting review" or "CVE severity>7 -> mandatory patch window" in other domains.

**Policy-gated decisions with causal chains and provenance** (Parts 10-11) is `AgentContext`, `PolicyEngine`, and `ProvenanceManager` gating a treatment change traced to a drug label here; the same classes gate a credit decision traced to a policy document, or an approval traced to a compliance filing, in another domain.

**GraphRAG over structured and unstructured data** (Part 12) is `AgentContext.retrieve` answering a clinical question from a label and the literature here; the identical call answers an investment question from filings and research notes elsewhere.

**Visualization and export to standard interchange formats** (Parts 13-14) is `KGVisualizer`, `OntologyVisualizer`, and `export_rdf`/`export_owl`/`export_neo4j_csv` rendering and exporting our clinical knowledge graph here, and the export layer is entirely domain-agnostic, so this step is identical no matter what the graph represents.

To adapt this notebook to a new domain, three things change and the rest stays the same: the real data sources in Part 2, the entity/relationship schema in Part 3, and the specific rule and policy text in Parts 6, 9, and 10. The ingestion, ontology, graph, search, reasoning, decision, provenance, visualization, and export mechanics are copy-paste reusable.

## Summary

Three patients, one evidence base built from six real sources, and fourteen parts that turned that evidence into an explainable clinical decision support system:

- **Part 1**: the case for structure and reasoning over pure vector-similarity RAG
- **Part 2**: real multi-source ingestion (Wikidata, MONDO/OLS4, PubMed, openFDA, RxNav, NLM Clinical Tables) with provenance
- **Part 3**: a medical ontology combining a real ingested ontology file with our own KG-derived classes
- **Part 4**: a SKOS vocabulary with real MONDO synonyms
- **Part 5**: alignment to real ICD-10-CM, MeSH, SNOMED CT, and UMLS identifiers
- **Part 6**: SHACL constraints grounded in real FDA text
- **Part 7**: the knowledge graph and context graph
- **Part 8**: clinical semantic search over real data
- **Part 9**: four independent reasoning engines, each explainable
- **Part 10**: causally-linked, policy-gated decisions with precedents
- **Part 11**: a provenance trail from decision to real regulatory source
- **Part 12**: GraphRAG, LLM-ready and LLM-optional
- **Part 13**: interactive visualization of graph, ontology, and decision timeline
- **Part 14**: export to RDF, OWL, JSON-LD, Neo4j CSV, and Parquet
- **Capstone**: all of it, run on a patient we hadn't met yet
- **Adapting this pattern**: the same 14-part structure applied to non-healthcare domains

### Next steps

- `02_Disease_Network_Analysis.ipynb`: a deeper dive into disease comorbidity networks
- `05_Medical_Database_Integration.ipynb`: live FHIR/MCP medical-database integration